In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("NER_dataset.csv", encoding="latin1")
df["Sentence #"] = df["Sentence #"].fillna(method="ffill")

/tmp/ipykernel_8980/3684407881.py:2: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df["Sentence #"] = df["Sentence #"].fillna(method="ffill")


In [5]:
sentences = df.groupby("Sentence #")["Word"].apply(list).values
pos_tags  = df.groupby("Sentence #")["POS"].apply(list).values

In [7]:
words = list(set(df["Word"].values))
tags  = list(set(df["POS"].values))

In [9]:
word2idx = {w: i+1 for i, w in enumerate(words)}
pos2idx  = {t: i   for i, t in enumerate(tags)}

In [11]:
idx2pos = {i:t for t,i in pos2idx.items()}


In [13]:
vocab_size = len(word2idx) + 1
n_tags = len(pos2idx)

In [15]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical


2026-02-24 11:01:14.552771: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 11:01:14.552967: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 11:01:14.581778: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 11:01:15.559572: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To tur

In [16]:
max_len = 50

In [19]:
X = [[word2idx[w] for w in s] for s in sentences]
y = [[pos2idx[t] for t in s] for s in pos_tags]

In [21]:
X = pad_sequences(X, maxlen=max_len, padding="post")
y = pad_sequences(y, maxlen=max_len, padding="post")


In [23]:
y = [to_categorical(i, num_classes=n_tags) for i in y]
y = np.array(y)

In [25]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)

In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, TimeDistributed, Dense

In [29]:
model = Sequential()
model.add(Embedding(input_dim=vocab_size,
                    output_dim=128,
                    input_length=max_len,
                    mask_zero=True))


/home/admin1/anaconda3/lib/python3.9/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [31]:
model.add(LSTM(units=128, return_sequences=True))


2026-02-24 11:01:23.209416: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [33]:
model.add(TimeDistributed(Dense(n_tags, activation="softmax")))

In [35]:
model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=["accuracy"])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [37]:
history = model.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=5,
    validation_data=(X_test, y_test)
)

Epoch 1/5


2026-02-24 11:01:29.772774: E tensorflow/core/util/util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.


338/338 ━━━━━━━━━━━━━━━━━━━━ 38s 107ms/step - accuracy: 0.2422 - loss: 1.8894 - val_accuracy: 0.4197 - val_loss: 0.1697
Epoch 2/5
338/338 ━━━━━━━━━━━━━━━━━━━━ 37s 109ms/step - accuracy: 0.4260 - loss: 0.1258 - val_accuracy: 0.4235 - val_loss: 0.1037
Epoch 3/5
338/338 ━━━━━━━━━━━━━━━━━━━━ 37s 109ms/step - accuracy: 0.4290 - loss: 0.0688 - val_accuracy: 0.4243 - val_loss: 0.0914
Epoch 4/5
338/338 ━━━━━━━━━━━━━━━━━━━━ 37s 109ms/step - accuracy: 0.4309 - loss: 0.0526 - val_accuracy: 0.4245 - val_loss: 0.0884
Epoch 5/5
338/338 ━━━━━━━━━━━━━━━━━━━━ 37s 109ms/step - accuracy: 0.4316 - loss: 0.0446 - val_accuracy: 0.4246 - val_loss: 0.0884


In [39]:
test_sentence = "London is a big city ."

In [41]:
def preprocess_sentence(sentence):
    seq = [word2idx.get(w, 0) for w in sentence.split()]
    seq = pad_sequences([seq], maxlen=max_len, padding="post")
    return seq

processed = preprocess_sentence(test_sentence)

In [43]:
prediction = model.predict(processed)
predicted_indices = np.argmax(prediction, axis=-1)[0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 303ms/step


In [45]:
tokens = test_sentence.split()

for token, idx in zip(tokens, predicted_indices[:len(tokens)]):
    print(token, "→", idx2pos[idx])

London → NNP
is → VBZ
a → DT
big → JJ
city → NN
. → .
